# logit-thermo 探索：場を足すと融点はどう動くか ——文章のクラウジウス・クラペイロンの下見（GPU、L4）

- **必要な secret：なし**（`HF_TOKEN` があれば取り込みが少し速くなるだけ）。**ランタイム：L4**（ハイメモリはいりません）。見込みは約 2 時間。終わったら自動で切れます。
- 探索（下見、予想なし）。写しの候補のトークンの logit に h（−0.3・0・＋0.3）を足して生成し、融点がどれだけ動くかを測る材料を集めます。モデルは Pythia-1B と OLMo-2-1B、温度 0.4〜0.7 の 7 個、サンプル 4。
- 計算のコード：A2-2・A2-4・A2-7・A2-7b のデータ集めのコードは Drive の前のノートブックから取り出してハッシュで確かめ、`scripts/a2_explore_field.py`（`0ac42c7ab82b06d2…`）は埋め込み。
- 保存先 `A2_explore/field/`。途中で切れても、もう一度「すべて実行」すれば終わった腕は飛ばします。**結果は表示しません。**
- **通知**：🚀・腕ごとの ✅・🎉・⚠️。名札は「logit-thermo 探索 場と融点（クラペイロンの下見、Pythia-1B・OLMo-1B、h ＝ −0.3・0・＋0.3、L4）」。終わったら、またはエラーで止まったら、ランタイムを切ります。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo 探索 場と融点（クラペイロンの下見、Pythia-1B・OLMo-1B、h ＝ −0.3・0・＋0.3、L4）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - explore field Clapeyron', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import torch
if not torch.cuda.is_available() or torch.cuda.get_device_properties(0).total_memory < 20e9:
    name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU なし'
    notify(f'⚠️ GPU のメモリが足りません（{name}）。L4 を選んでください。ランタイムを切ります', tags='warning')
    from google.colab import runtime; runtime.unassign()
    raise SystemExit
torch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision('highest')
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：計算のコード
import ast, glob, hashlib, json
def take_src(want):
    """want：{変数の名前: sha256}。Drive の logit-thermo のノートブックから、埋め込んだコードを取り出してハッシュで確かめる。"""
    got = {}
    for p in sorted(glob.glob('/content/drive/MyDrive/logit-thermo' + '/**/*.ipynb', recursive=True)):
        try:
            cells = json.load(open(p))['cells']
        except Exception:
            continue
        for c in cells:
            s = c['source'] if isinstance(c['source'], str) else ''.join(c['source'])
            if c.get('cell_type') != 'code' or 'SRC' not in s:
                continue
            try:
                body = ast.parse(s).body
            except SyntaxError:
                continue
            for n in body:
                if isinstance(n, ast.Assign) and isinstance(n.targets[0], ast.Name) and n.targets[0].id in want and n.targets[0].id not in got:
                    v = ast.literal_eval(n.value)
                    if hashlib.sha256(v.encode()).hexdigest() == want[n.targets[0].id]:
                        got[n.targets[0].id] = v
        if len(got) == len(want):
            break
    miss = [k for k in want if k not in got]
    assert not miss, ('Drive のノートブックからコードを取り出せませんでした（前のノートブックを動かしていませんか）：', miss)
    return got
_s = take_src({'SRC22': 'e578cfc480980be6b0d1e18377b6beb8b7555e0744f56b13f63cb304a9f73d53', 'SRC24': '6ef87866ae7d895d63c6e07d48a58a2ddf4111a4802ffe98dbe7f61892c96b57', 'SRC7C': '8f8776cbe4a0e1a7b1f676c4f84ba70fb9c6c0b46ef1466969e09f0ba062b550', 'SRC7BC': 'b03ff9e2b16e2297fb9142a26e15f561caacf77216453d78e2024066c01c57d0'})
SRC22, SRC24, SRC7C, SRC7BC = _s['SRC22'], _s['SRC24'], _s['SRC7C'], _s['SRC7BC']
SRCF = '"""探索 8.30（下見）：「場」を足すと融点はどう動くか ——文章のクラウジウス・クラペイロン（2026-10-31 09:4x、GPU（L4）、判定しない）。\nscripts/a2_2_collect.py・a2_4_collect.py・a2_7_collect.py・a2_7b_collect.py を実行したあとの同じ名前空間で実行する（load_model・valid_mask・\nsplit_doc・load_docs・CopyIndexN・B7_N・CONT・SAMPLES を変えずに使う）。\n\n場 h：各歩で、3 トークンの写しの候補（直前 2 トークンのあとに、それまでに続いたトークン）の logit に h を足す（T で割る前）。\nh > 0 は結晶寄り、h < 0 は液体寄り。熱力学では、エネルギーに −h × M（M ＝ 写しの候補を選んだ歩の数）を足すことにあたる。\n熱力学どおりなら、融点の動き dT_m/dh ＝ ΔM ÷ ΔS（ΔM：1 回の相転移で変わる M、ΔS：1 回の相転移のエントロピーの差）。\nΔM は、液体と結晶で「写しの候補を選んだ割合」の差 × 協同性の単位 n（A2-8）からも見積もれるので、2 通りで比べられる。\n\n腕：h ＝ −0.3・0・＋0.3（logit の単位、ナット）。温度 0.4〜0.7 の 7 個。サンプル 4（種は h によらず同じ：同じ乱数で比べる）。\n保存：トークン、写しの候補があったか、選んだトークンが写しの候補だったか。\n"""\nimport gc, os, time\nimport numpy as np\nimport torch\n\nF_TEMPS = [0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7]\nF_H = [(\'hm03\', -0.3), (\'h0\', 0.0), (\'hp03\', 0.3)]\nF_S = 4\nF_SEED = 20261127\nF_MODELS = [(\'pythia-1b\', \'EleutherAI/pythia-1b\'), (\'olmo2-1b\', \'allenai/OLMo-2-0425-1B\')]\n\n\ndef f_seed(k, s):\n    return F_SEED + 1000 * k + s\n\n\n@torch.no_grad()\ndef generate_field(model, prompts, T, valid, gen, h):\n    device = valid.device\n    B = len(prompts)\n    x = torch.tensor(prompts, device=device)\n    out = model(x, use_cache=True)\n    past, logits = out.past_key_values, out.logits[:, -1, :].float()\n    idx = [CopyIndexN(p, B7_N) for p in prompts]\n    toks, cand, pick = [], [], []\n    for t in range(CONT):\n        z = logits.double().masked_fill(~valid, float(\'-inf\'))\n        cands = [idx[b].candidates() for b in range(B)]\n        has = torch.tensor([len(c) > 0 for c in cands], device=device)\n        if h != 0.0:\n            rows, cols = [], []\n            for b, c in enumerate(cands):\n                rows += [b] * len(c); cols += list(c)\n            if rows:\n                z[torch.tensor(rows, device=device), torch.tensor(cols, device=device)] += h\n        p = torch.softmax(z / T, dim=-1)\n        nxt = torch.multinomial(p, 1, generator=gen)\n        pk = torch.tensor([int(nxt[b, 0]) in set(cands[b]) for b in range(B)], device=device)\n        for b in range(B):\n            idx[b].push(int(nxt[b, 0]))\n        toks.append(nxt[:, 0]); cand.append(has); pick.append(pk)\n        out = model(nxt, past_key_values=past, use_cache=True)\n        past, logits = out.past_key_values, out.logits[:, -1, :].float()\n    st = lambda v: torch.stack(v, 1).cpu().numpy()\n    return dict(toks=st(toks).astype(np.int32), cand=st(cand).astype(np.bool_), pick=st(pick).astype(np.bool_))\n\n\ndef run_field(name, hf_id, docs, outdir, device, notify=print):\n    todo = [(a, h) for a, h in F_H if not os.path.exists(f\'{outdir}/{name}@{a}.npz\')]\n    if not todo:\n        print(name, \'はもう全部終わっています（飛ばします）\')\n        return\n    t0 = time.time()\n    tok, model = load_model(hf_id, device)\n    assert next(model.parameters()).dtype == torch.float32\n    valid, eos = valid_mask(tok, model, device)\n    pairs = [split_doc(tok, d[\'text\']) for d in docs]\n    use = [i for i, p in enumerate(pairs) if p is not None]\n    nD = len(use)\n    assert len({len(pairs[i][0]) for i in use}) == 1\n    prompts = [pairs[i][0] for i in use]\n    for arm, h in todo:\n        ta = time.time()\n        R = {k: np.zeros((len(F_TEMPS), F_S, nD, CONT), dt) for k, dt in ((\'toks\', np.int32), (\'cand\', np.bool_), (\'pick\', np.bool_))}\n        for k, T in enumerate(F_TEMPS):\n            for s in range(F_S):\n                g = torch.Generator(device=device).manual_seed(f_seed(k, s))\n                r = generate_field(model, prompts, T, valid, g, h)\n                for kk in R:\n                    R[kk][k, s] = r[kk]\n        path = f\'{outdir}/{name}@{arm}.npz\'\n        np.savez_compressed(path + \'.part.npz\', temps=np.array(F_TEMPS), h=h, use=np.array(use), seconds=time.time() - ta, **R)\n        os.replace(path + \'.part.npz\', path)\n        notify(f\'{name}@{arm} おわり（{time.time() - ta:.0f} 秒、通算 {time.time() - t0:.0f} 秒）\')\n    del model; gc.collect(); torch.cuda.empty_cache()\n\n\n@torch.no_grad()\ndef self_test_field(device):\n    """小さなランダムのモデルで：h ＝ 0 は A2-7b の「なし」と同じトークン。h が大きいと、候補がある歩ではほぼいつも候補を選ぶ。"""\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    hf_id = \'hf-internal-testing/tiny-random-GPT2LMHeadModel\'\n    tok = AutoTokenizer.from_pretrained(hf_id)\n    model = AutoModelForCausalLM.from_pretrained(hf_id, torch_dtype=torch.float32).to(device).eval()\n    valid, _ = valid_mask(tok, model, device)\n    global CONT\n    keep = CONT\n    CONT = 80\n    try:\n        prompts = [[5, 6, 7, 8] * 4, [9, 10, 11, 12] * 4, [3, 4, 3, 4] * 4, [20, 21, 22, 23] * 4]\n        a = generate_field(model, prompts, 0.7, valid, torch.Generator(device=device).manual_seed(3), 0.0)\n        b = generate_7b(model, prompts, 0.7, valid, torch.Generator(device=device).manual_seed(3), 33, \'none\', 0.0, np.random.default_rng(0))\n        assert (a[\'toks\'] == b[\'toks\']).all(), \'h ＝ 0 が「なし」と違う\'\n        c = generate_field(model, prompts, 0.7, valid, torch.Generator(device=device).manual_seed(4), 30.0)\n        assert c[\'pick\'][c[\'cand\']].mean() > 0.99, c[\'pick\'][c[\'cand\']].mean()\n        d = generate_field(model, prompts, 0.7, valid, torch.Generator(device=device).manual_seed(4), -30.0)\n        assert d[\'pick\'].mean() < 0.01, d[\'pick\'].mean()\n    finally:\n        CONT = keep\n    assert len(F_TEMPS) == 7 and [a for a, _ in F_H] == [\'hm03\', \'h0\', \'hp03\'] and F_S == 4\n    return \'OK\'\n'
assert hashlib.sha256(SRCF.encode()).hexdigest() == '0ac42c7ab82b06d2e7443ba9c0c0e147c0121b5351e0b5830b1a54b46043190d'
exec(SRC22)
exec(SRC24)
exec(SRC7C)
exec(SRC7BC)
exec(SRCF)
print('コードを置きました')


In [ ]:
# セル3：自己テスト → 記事（A2-2 と同じか確かめる）→ モデルごとに 3 本の腕 → 保存（結果は表示しない）
import json, sys, transformers
try:
    print('自己テスト：', self_test_a27('cuda'), self_test_7b('cuda'), self_test_field('cuda'))
    os.makedirs('/content/drive/MyDrive/logit-thermo/A2_explore/field', exist_ok=True)
    assert SEED == 20261027, ('記事を選ぶ種が A2-2 と違います', SEED)
    docs, meta = load_docs()
    old = json.load(open('/content/drive/MyDrive/logit-thermo' + '/A2_2/docs_meta.json'))
    assert [d['text_sha256'] for d in old['docs']] == [d['text_sha256'] for d in meta['docs']], 'BBC の記事が A2-2 と違います'
    for name, hf_id in F_MODELS:
        run_field(name, hf_id, docs, '/content/drive/MyDrive/logit-thermo/A2_explore/field', 'cuda', notify=lambda s: notify('✅ ' + s, tags='white_check_mark'))
    json.dump(dict(models=F_MODELS, temps=F_TEMPS, h=F_H, samples=F_S, seed=F_SEED, a2_7b_collect_sha256='b03ff9e2b16e2297fb9142a26e15f561caacf77216453d78e2024066c01c57d0',
                   field_sha256='0ac42c7ab82b06d2e7443ba9c0c0e147c0121b5351e0b5830b1a54b46043190d', torch=torch.__version__, transformers=transformers.__version__, python=sys.version.split()[0],
                   gpu=torch.cuda.get_device_name(0), matmul=torch.get_float32_matmul_precision()),
              open('/content/drive/MyDrive/logit-thermo/A2_explore/field' + '/run_meta.json', 'w'), ensure_ascii=False, indent=1)
    print('保存しました')
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ 探索 場と融点（クラペイロンの下見）で止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify('🎉 終わりました：場と融点の下見の材料を A2_explore/field に保存しました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
